# Analog Neural Network with Classes

## Random Functionality

In [1]:
import random

In [2]:
def random_weight() -> float:
    return random.uniform(-1.0, 1.0)

## Gradient Functionality

In [3]:
small_step = 1e-9

In [4]:
def variation_rate(f, x: float) -> float:
    return (f(x + small_step) - f(x - small_step)) / (2 * small_step)

## Basic Classes:

In [5]:
class Neuron:
    def __init__(self, W: list[float], b: float, F = lambda x: x) -> None:
        self.W = W
        self.b = b
        self.F = F

        self.saved_X = []
        self.saved_logit = 0.0
        self.saved_score = 0.0
        
        self.dW = 0.0
        self.db = 0.0

    @classmethod
    def random(cls, n_inputs: int, F = lambda x: x) -> Neuron:
        return cls([random_weight() for _ in range(n_inputs)], random_weight(), F)

In [6]:
class Neuron(Neuron):
    def get_n_inputs(self) -> int:
        return len(self.W)
        
    def get_W(self) -> list[float]:
        return self.W
        
    def set_W(self, W: list[float]) -> None:
        self.W = W
        
    def get_b(self) -> float:
        return self.b
        
    def set_b(self, b: float) -> None:
        self.b = b

In [7]:
class Neuron(Neuron):
    def score(self, X: list[float], save_score: bool = False) -> float:
        logit = sum([x * w for x, w in zip(X, self.W)]) + self.b
        score = self.F(logit)
        if save_score:
            self.saved_X = X
            self.saved_logit = logit
            self.saved_score = score
        return score

    def __call__(self, X: list[float]) -> float:
        return self.score(X)

In [8]:
class Neuron(Neuron):
    def zero_grad(self) -> None:
        self.dW = 0.0
        self.db = 0.0

    def backward(self, upstream_grad: float) -> list[float]:
        downstream_grads = [0.0 for _ in range(self.get_n_inputs())]
        delta = variation_rate(self.F, self.last_logit) * upstream_grad
        self.db += delta
        for i, (x, w) in enumerate(zip(self.saved_X, self.W)):
            self.dW[i] += delta * x
            downstream_grads[i] += delta * w
        return downstream_grads

    def step(self, learning_rate: float) -> None:
        self.W = [w - dw * learning_rate for w, dw in zip(self.W, self.dW)]
        self.b -= self.db * learning_rate

In [9]:
class Layer:
    def __init__(self, neurons = list[Neuron]) -> None:
        self.neurons = neurons

    @classmethod
    def random(cls, n_inputs: int, n_outputs: int, F = lambda x: x) -> Layer:
        return cls([Neuron.random(n_inputs, F) for _ in range(n_outputs)])

In [10]:
class Layer(Layer):
    def get_n_neurons(self) -> int:
        return len(self.neurons)

    def get_n_inputs(self) -> int:
        return len(self.neurons[0].get_W())

    def get_neurons(self) -> list[Neuron]:
        return self.neurons

    def set_neurons(self, neurons: list[Neuron]) -> None:
        self.neurons = neurons

In [11]:
class Layer(Layer):
    def score(self, X: list[float], save_score: bool = False) -> list[float]:
        scores = [neuron.score(X, save_score) for neuron in self.neurons]
        return scores
        
    def __call__(self, X: list[float]) -> list[float]:
        return self.score(X)

In [12]:
class Layer(Layer):
    def zero_grad(self) -> None:
        for neuron in self.neurons:
            neuron.zero_grad()

    def backward(self, upstream_grads: list[float]) -> list[float]:
        downstream_grads = [0.0 for _ in range(self.get_n_inputs())]
        for neuron, upstream_grad in zip(self.neuron, upstream_grads):
            downstream_grads = [downstream_grad + delta for downstream_grad, delta in zip(downstream_grads, neuron.backwards(downstream_grad))]
        return downstream_grads

    def step(self, learning_rate: float) -> None:
        for neuron in self.neurons:
            neuron.step(learning_rate)

In [13]:
class Network:
    def __init__(self, layers = list[Layer]) -> None:
        self.layers = layers

    @classmethod
    def random(cls, shape: list[int], Fs : list = None) -> Network:
        if Fs is None:
            Fs = [lambda x: x for _ in range(1, len(shape))]
        return cls([Layer.random(shape[i - 1], shape[i], Fs[i - 1]) for i in range(1, len(shape))])

In [14]:
class Network(Network):
    def get_shape(self) -> list[int]:
        return [self.layers[0].get_n_inputs()] + [layer.get_n_neurons for layer in self.layers]
        
    def get_layers(self) -> list[Layer]:
        return self.layers

    def set_layers(self, layers: list[Layer]) -> None:
        self.layers = layers

In [15]:
class Network(Network):
    def score(self, X: list[float], save_score: bool = False) -> list[float]:
        for layer in self.layers:
            X = layer.score(X, save_score)
        scores = X
        return scores
        
    def __call__(self, X: list[float]) -> list[float]:
        return self.score(X)

In [16]:
class Network(Network):
    def zero_grad(self) -> None:
        for layer in self.layers:
            layer.zero_grad()

    def backward(self, loss_grads: list[float]) -> None:
        grad = loss_grads
        for layer in reversed(self.layers):
            grad = layer.backward(grad)
            
    def step(self, learning_rate: float) -> None:
        for layer in self.layers:
            layer.step(learning_rate)

## Activation functions

In [17]:
import math

In [18]:
def sigmoid(x: float) -> float:
    exp = math.exp(x)
    return exp / (exp + 1.0)

## Test

In [19]:
nn = Network.random([4, 8, 4, 2, 1], [sigmoid, sigmoid, sigmoid, sigmoid])

In [20]:
for _ in range(8):
    X = [random_weight() for _ in range(4)]
    print(X, nn(X))

[-0.2960132957814807, -0.21101078839164433, -0.6079550303222725, -0.7403522263802171] [0.3762805379580785]
[-0.8226961476090504, -0.2663755803450334, -0.1674805522174505, -0.13150525836708993] [0.3765395993655201]
[-0.1885652770443702, 0.3268720123370574, 0.3533122406228584, -0.2497376918233225] [0.37901449659610426]
[0.8681123885549702, 0.9964003609580632, 0.40406549831602545, -0.5955506310290326] [0.3816616371119509]
[-0.9213361188978926, 0.9198990563598213, 0.9197182547109353, -0.9533894730069428] [0.3831401032723218]
[0.8587256529874789, 0.23441040005101943, 0.6528337330816771, 0.9414323273505312] [0.3777429416029256]
[0.5306275162880691, 0.6063971913270538, -0.7327396877394836, -0.5857255844587315] [0.3814399608164197]
[-0.35086695390258527, 0.028858867493180895, -0.4009715694898668, -0.24875474673910003] [0.3782303927017797]
